In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" 

In [2]:
x = torch.randn(100, 16, 784)
layer = nn.BatchNorm1d(16) #number of channels/layer nomalization 
out = layer(x) #x forward
layer.running_mean
layer.running_var


tensor([-1.8619e-04, -4.2718e-04,  5.9558e-04, -9.1018e-05,  1.1944e-05,
        -1.1676e-04, -1.7546e-04,  1.4749e-04, -8.6933e-05, -3.6135e-04,
         3.6393e-04,  6.0442e-04,  1.6533e-04,  2.5274e-04,  1.9848e-04,
        -3.8059e-04])

tensor([1.0006, 1.0001, 1.0007, 1.0002, 0.9997, 1.0006, 0.9991, 1.0008, 1.0005,
        1.0002, 1.0007, 0.9998, 1.0001, 0.9990, 1.0002, 1.0005])

In [3]:
x = torch.randn(100, 16, 7, 7, requires_grad=True) 
layer = nn.BatchNorm2d(16, affine=True) #number of channels/layer nomalization
out = layer(x) #x forward
out.shape
layer.running_mean
layer.running_var
layer.weight
layer.bias

torch.Size([100, 16, 7, 7])

tensor([-1.2243e-04,  1.8369e-04,  7.4832e-04,  2.0629e-03,  1.4540e-03,
        -5.0540e-04, -1.0596e-03,  5.2022e-05,  3.0018e-04,  9.7941e-04,
        -1.7949e-03,  1.9471e-03, -4.0826e-04,  2.8469e-03,  1.5237e-03,
         1.3691e-03])

tensor([0.9991, 0.9999, 1.0004, 0.9994, 0.9993, 1.0004, 0.9986, 1.0001, 1.0011,
        0.9992, 0.9977, 1.0020, 1.0023, 1.0028, 0.9995, 0.9943])

Parameter containing:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       requires_grad=True)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       requires_grad=True)

In [ ]:
class ResBlk(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlk, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1) #convert input date into out size
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        #ensure residantial have the same dimesion with input
        if in_channels != out_channels:
            self.extra = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=1),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.extra = nn.Sequential()
            
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.extra(x)
        out = F.relu(out)
        return out
    
myS = nn.Sequential(
    nn.Linear(784, 200),
    ResBlk(200, 200),
    nn.ReLU(),
    nn.Linear(200, 100),
    nn.ReLU(),
    nn.Linear(100, 10)
)
# myS.to(device)

list(myS.named_parameters()) #use .parameters to optimize the parameters:dirct input

#save and load
myS.load_state_dict(torch.load('mnist_cnn.pth'))
myS.save(myS.state_dict(), 'mnist_cnn.pth')

#train and test
myS.train()
myS.eval()

[('0.weight',
  Parameter containing:
  tensor([[ 0.0079, -0.0116, -0.0196,  ...,  0.0219, -0.0075,  0.0203],
          [ 0.0324,  0.0333, -0.0192,  ...,  0.0097, -0.0095,  0.0257],
          [-0.0124, -0.0225,  0.0011,  ...,  0.0028, -0.0350,  0.0045],
          ...,
          [ 0.0277, -0.0138, -0.0139,  ...,  0.0189,  0.0008, -0.0340],
          [ 0.0251, -0.0120, -0.0223,  ..., -0.0064, -0.0165, -0.0325],
          [-0.0102, -0.0088, -0.0309,  ...,  0.0049, -0.0309, -0.0316]],
         requires_grad=True)),
 ('0.bias',
  Parameter containing:
  tensor([ 3.3585e-02, -3.4797e-03,  1.1249e-02, -6.5091e-04, -1.8548e-02,
           1.6947e-02,  1.5999e-02, -1.6435e-02,  3.0345e-03,  1.1202e-02,
           1.2969e-02, -2.7390e-02, -3.1994e-02,  6.5364e-03, -8.6184e-03,
          -1.8465e-02, -2.9861e-02, -3.3087e-02, -9.0720e-03,  1.6658e-02,
           8.1398e-03,  2.1786e-02, -5.2135e-03,  3.3625e-02,  2.8752e-02,
           4.4170e-03,  1.7807e-02,  2.3079e-02, -1.9344e-02, -2.6711e-0

In [13]:
#nn moudule
'''
sequential:container of layers
'''
import torch
import torch.nn as nn

class MyCustomLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MyCustomLayer, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        # 可以在这里加上自己的逻辑，比如激活函数、变换、归一化等
        return self.linear(x)
    
model = nn.Sequential(
    MyCustomLayer(16, 64),   # 自定义层
    nn.ReLU(),               # 激活函数
    nn.Linear(64, 10),       # 输出层
    nn.Softmax(dim=1)        # 输出归一化
)

list(model.parameters())


'\nsequential:container of layers\n'

[Parameter containing:
 tensor([[-0.1213, -0.0332,  0.1919,  ...,  0.0378,  0.1688,  0.0617],
         [ 0.0310,  0.0986,  0.0510,  ..., -0.0131, -0.1488, -0.1198],
         [-0.1167, -0.1227, -0.1029,  ...,  0.2030, -0.0948, -0.0219],
         ...,
         [ 0.0267, -0.1644,  0.0837,  ..., -0.1131, -0.2117,  0.0972],
         [ 0.1883,  0.0971, -0.2282,  ..., -0.0235, -0.2359, -0.0403],
         [ 0.1103,  0.0559, -0.1848,  ...,  0.0016,  0.2446,  0.2261]],
        requires_grad=True),
 Parameter containing:
 tensor([-3.2646e-02,  2.1564e-01, -1.3405e-01, -1.2126e-01,  2.3902e-01,
         -8.6312e-03, -1.0171e-01,  2.0261e-01, -1.4055e-01,  1.5187e-01,
          6.2784e-02,  3.1184e-02,  1.9905e-02, -9.3410e-02,  5.9611e-02,
         -1.4367e-01, -4.6519e-02, -2.3248e-01, -7.9392e-02, -2.3261e-01,
         -1.7101e-01, -1.1906e-01,  2.3371e-01, -2.1256e-01, -2.2005e-01,
          1.0136e-01, -1.9053e-01, -7.1563e-02, -1.9679e-01, -2.9155e-02,
          2.1154e-01, -2.1704e-01,  2.48

In [ ]:
class Flatten(nn.Module):
    def __init__(self):
        super(Flatten, self).__init__()
        
    def forward(self, input):
        return input.view(input.size(0), -1)

#own class
class MyLinear(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MyLinear, self).__init__()
        self.w = nn.parameter(torch.randn(input_dim, output_dim)) #applied .parameters function
        self.b = nn.parameter(torch.randn(output_dim))
        
    def forward(self, x):
        return torch.matmul(x, self.w) + self.b
        